# 面试问题：ORPO 如何把 chosen SFT NLL 与 rejected odds-ratio penalty 放在单阶段训练中，并且不需要 reference model？

**一句话回答。** ORPO（Odds Ratio Preference Optimization）直接在同一个策略模型上优化 `chosen NLL + λ × odds-ratio penalty`：chosen 响应继续承担常规监督微调，让模型学习任务和语言分布；偏好项比较 chosen 与 rejected 序列概率对应的 odds，拉高 chosen 相对 rejected 的对数优势。它不需要冻结 reference model，也不需要先 SFT 再做第二阶段偏好优化，但仍必须对序列概率的数值稳定、长度归一化、梯度方向、数据配对质量与离线门禁负责。

本 Notebook 不调用 Trainer 或现成偏好优化器，而是用基础 PyTorch 手写稳定 `log1mexp`、log-odds、OR penalty、组合损失、梯度检查和 `TinyLM.forward`，再从同一初始化比较纯 SFT 与 ORPO。示例用于解释机制，不能替代真实 LLM 上的分布式训练、基准评测和安全审计。

**主要资料。** [ORPO: Monolithic Preference Optimization without Reference Model](https://arxiv.org/abs/2403.07691)。

In [ ]:
import copy  # 导入深拷贝工具，保证 SFT 与 ORPO 从完全相同的权重开始。
import math  # 导入自然对数常数与数值比较所需的数学函数。
import torch  # 导入 PyTorch 张量与自动微分基础能力。
from torch import nn  # 导入神经网络模块基类以手写 TinyLM。
torch.manual_seed(243)  # 固定随机种子，使初始化、训练轨迹和断言可复现。
device = torch.device("cpu")  # 固定使用 CPU，确保普通开发环境可冷启动执行。
default_dtype = torch.float64  # 使用双精度验证接近零概率边界的数值稳定性。
assert torch.__version__  # 验证运行环境已经正确加载 PyTorch。
assert torch.initial_seed() == 243  # 验证当前实验确实采用约定随机种子。
assert device.type == "cpu"  # 验证示例没有隐藏依赖 GPU。
assert default_dtype == torch.float64  # 验证稳定性探针会使用足够精度。

## 1. 先说清 ORPO 到底合并了什么

给定 prompt `x`、偏好响应 `y_w` 和拒绝响应 `y_l`，第一部分是 chosen 的监督负对数似然：`L_SFT = -log pθ(y_w|x)`。第二部分先把序列概率转换为 odds：`odds(p)=p/(1-p)`，再比较 `log odds_w - log odds_l`，用 `-log sigmoid(margin)` 惩罚错误排序。最终目标是 `L = L_SFT + λ L_OR`。

这叫“单阶段”不是因为只有一个数学项，而是两个项对同一个可训练模型、同一批 chosen/rejected 配对进行一次前向反向。SFT 项维持生成学习信号，OR 项显式使用 rejected 形成排序边界。工程账本仍要分别记录两项，否则总 loss 下降时无法判断究竟是语言建模改善，还是偏好项在主导。

In [ ]:
def stable_log1mexp(log_probability):  # 定义稳定计算 log(1-exp(log_probability)) 的分段函数。
    assert torch.all(log_probability <= 0.0)  # 验证输入确实是不会大于零的对数概率。
    epsilon = torch.finfo(log_probability.dtype).eps  # 取得当前浮点类型可表达的机器精度。
    safe_log_probability = torch.clamp(log_probability, max=-epsilon)  # 避免概率恰好为一时出现无定义对数。
    threshold = -math.log(2.0)  # 使用经典分界点减少两个分支各自的消减误差。
    far_from_zero = torch.log1p(-torch.exp(safe_log_probability))  # 在较小概率区间使用 log1p 保留尾部精度。
    near_zero = torch.log(-torch.expm1(safe_log_probability))  # 在概率接近一时使用 expm1 避免一减相近数。
    return torch.where(safe_log_probability < threshold, far_from_zero, near_zero)  # 按输入区间选择更稳定的结果。
probe_log_probabilities = torch.tensor([-20.0, -1.0, -0.1, -1e-10], dtype=default_dtype)  # 构造从极小概率到接近一的边界样本。
stable_complements = stable_log1mexp(probe_log_probabilities)  # 计算所有探针的对数补概率。
direct_middle = torch.log(1.0 - torch.exp(probe_log_probabilities[1:3]))  # 在安全的中间区域计算直接公式作为参照。
assert torch.isfinite(stable_complements).all()  # 验证极端输入仍然得到有限结果。
assert torch.allclose(stable_complements[1:3], direct_middle, atol=1e-12)  # 验证稳定实现与直接公式在安全区一致。
assert torch.all(stable_complements[:-1] > stable_complements[1:])  # 验证原概率越大时补概率的对数越小。
assert abs(float(stable_complements[0])) < 1e-8  # 验证极小原概率对应的补概率接近一。
assert stable_complements[-1] < -20.0  # 验证接近一的原概率会产生很小的补概率。

## 2. 为什么 `log(1-p)` 必须稳定实现

直接先做 `p=exp(log_p)` 再做 `log(1-p)`，当 `p` 接近 1 时会发生严重消减：浮点表示可能把 `p` 舍入成 1，随后得到负无穷。ORPO 的 odds 恰好包含 `1-p`，因此高置信 chosen 样本最容易触发这个问题。稳定实现以 `-log(2)` 为界：远离零时使用 `log1p(-exp(x))`，靠近零时使用 `log(-expm1(x))`。

生产中还要明确序列概率的定义。长序列的概率乘积天然更小，若直接比较概率总和，会把长度差异混入偏好信号。本例默认对有效 token 的 log probability 求均值，使不同长度样本更可比；若复现某份论文或代码采用求和，则必须固定实现并做长度分桶评测，不能悄悄切换。

In [ ]:
def stable_softplus(value):  # 用基础张量算子手写稳定 softplus 以实现负对数 sigmoid。
    return torch.clamp(value, min=0.0) + torch.log1p(torch.exp(-torch.abs(value)))  # 通过绝对值形式避免大正数指数溢出。
def log_odds_from_log_probability(log_probability):  # 把序列对数概率转换成稳定的对数 odds。
    return log_probability - stable_log1mexp(log_probability)  # 根据 log(p/(1-p)) 展开为两个稳定对数项。
def odds_ratio_penalty(chosen_log_probability, rejected_log_probability):  # 定义 ORPO 的逐样本 odds-ratio 偏好惩罚。
    chosen_log_odds = log_odds_from_log_probability(chosen_log_probability)  # 计算 chosen 响应的对数优势。
    rejected_log_odds = log_odds_from_log_probability(rejected_log_probability)  # 计算 rejected 响应的对数优势。
    preference_margin = chosen_log_odds - rejected_log_odds  # 计算 chosen 相对 rejected 的对数优势差。
    return stable_softplus(-preference_margin)  # 计算 -log(sigmoid(margin)) 且保持数值稳定。
good_chosen = torch.tensor([-0.2], dtype=default_dtype)  # 构造概率较高的 chosen 响应。
bad_rejected = torch.tensor([-2.0], dtype=default_dtype)  # 构造概率较低的 rejected 响应。
good_penalty = odds_ratio_penalty(good_chosen, bad_rejected)  # 计算排序正确时的偏好惩罚。
swapped_penalty = odds_ratio_penalty(bad_rejected, good_chosen)  # 计算排序颠倒时的偏好惩罚。
equal_penalty = odds_ratio_penalty(good_chosen, good_chosen)  # 计算双方概率相同时的基准惩罚。
assert log_odds_from_log_probability(good_chosen) > log_odds_from_log_probability(bad_rejected)  # 验证概率越高则对数 odds 越大。
assert good_penalty < math.log(2.0)  # 验证正确排序的惩罚低于随机边界。
assert swapped_penalty > math.log(2.0)  # 验证错误排序的惩罚高于随机边界。
assert torch.allclose(equal_penalty, torch.tensor([math.log(2.0)], dtype=default_dtype))  # 验证零 margin 对应 log(2)。
assert torch.allclose(stable_softplus(torch.tensor([0.0], dtype=default_dtype)), equal_penalty)  # 验证手写 softplus 的零点正确。
assert torch.isfinite(odds_ratio_penalty(torch.tensor([-1e-10], dtype=default_dtype), bad_rejected)).all()  # 验证高置信 chosen 不会导致非数值。

## 3. odds ratio 比普通概率差多表达了什么

概率差 `p_w-p_l` 在接近零或一时尺度会压缩；odds `p/(1-p)` 同时看“选择该响应”和“不选择该响应”的相对可能性。ORPO 再比较两边的 log-odds，把乘除变成加减，并用 logistic 形式形成平滑排序损失。margin 为零时 penalty 是 `log(2)`；chosen 更占优时 penalty 下降；顺序颠倒时 penalty 上升。

这里不能把 odds 当成推理阶段的概率校准承诺。它是训练目标中的相对量，最终模型仍需在独立数据上检查胜率、长度偏差、KL 漂移、事实性、安全性和过度拒答。偏好对本身若带有模板、长度或标注者偏差，ORPO 也会学习这些捷径。

In [ ]:
class TinyLM(nn.Module):  # 定义一个显式实现 forward 的最小条件语言模型。
    def __init__(self, prompt_count, vocabulary_size, hidden_size, maximum_length):  # 接收提示数、词表、隐层和最大响应长度。
        super().__init__()  # 初始化模块基类以登记参数和子模块。
        self.prompt_embedding = nn.Embedding(prompt_count, hidden_size)  # 为每个玩具 prompt 学习条件表示。
        self.position_embedding = nn.Embedding(maximum_length, hidden_size)  # 为每个响应位置学习位置表示。
        self.output_projection = nn.Linear(hidden_size, vocabulary_size)  # 把隐状态投影为下一 token logits。
    def forward(self, prompt_ids, sequence_length):  # 根据 prompt 与响应长度计算逐位置词表分数。
        positions = torch.arange(sequence_length, device=prompt_ids.device)  # 构造从零开始的响应位置编号。
        prompt_states = self.prompt_embedding(prompt_ids).unsqueeze(1)  # 把 prompt 表示扩展出序列维度。
        position_states = self.position_embedding(positions).unsqueeze(0)  # 把位置表示扩展出批次维度。
        hidden_states = torch.tanh(prompt_states + position_states)  # 融合条件与位置信息并施加非线性。
        return self.output_projection(hidden_states)  # 返回批次、序列、词表三维 logits。
prompt_ids = torch.tensor([0, 1, 2, 3], dtype=torch.long, device=device)  # 构造四条偏好样本的 prompt 编号。
chosen_tokens = torch.tensor([[1, 2, 0], [2, 3, 4], [3, 4, 0], [4, 5, 6]], dtype=torch.long, device=device)  # 构造标注者偏好的响应 token。
rejected_tokens = torch.tensor([[2, 1, 0], [3, 2, 5], [4, 3, 0], [5, 4, 1]], dtype=torch.long, device=device)  # 构造与 chosen 配对的拒绝响应 token。
response_mask = torch.tensor([[1, 1, 0], [1, 1, 1], [1, 1, 0], [1, 1, 1]], dtype=torch.bool, device=device)  # 标记真实 token 并屏蔽补齐位置。
template_model = TinyLM(prompt_count=4, vocabulary_size=7, hidden_size=12, maximum_length=3).to(device)  # 创建后续实验共享初始化的 TinyLM。
template_logits = template_model(prompt_ids, sequence_length=3)  # 执行一次前向传播检查模型契约。
initial_state = copy.deepcopy(template_model.state_dict())  # 冻结初始化权重供不同训练策略公平复用。
assert template_logits.shape == torch.Size([4, 3, 7])  # 验证输出维度符合批次、响应位置、词表约定。
assert response_mask.sum().item() == 10  # 验证四条响应一共包含十个有效 token。
assert torch.all(chosen_tokens[response_mask] != rejected_tokens[response_mask])  # 验证每个有效位置都提供真实偏好对比。
assert set(initial_state) == set(template_model.state_dict())  # 验证冻结快照包含模型全部参数。
assert sum(parameter.numel() for parameter in template_model.parameters()) > 0  # 验证手写模型确实具有可训练参数。

## 4. TinyLM 与偏好批次怎样对应真实训练

`TinyLM` 用 prompt embedding 加 position embedding 产生每个响应位置的 logits，并显式实现 `forward`。它没有注意力与自回归历史，只保留 ORPO 最关键的训练接口：同一 prompt 下能计算 chosen 和 rejected 序列的条件 log probability，梯度能回到同一组策略参数。真实 LLM 会把 prompt 与响应拼接，并只在响应 token 上计算 mask 后的 log probability。

配对数据必须保证 prompt 完全一致、chosen/rejected 没有错位，padding 不进入 loss，截断没有只删掉某一侧的关键结论。数据管道还应记录来源、标注规范、长度、语言、安全类别和去重簇；否则实现公式正确也可能把脏偏好放大。

In [ ]:
def manual_log_softmax(logits):  # 用基础算子手写数值稳定的 log-softmax。
    row_maximum = logits.max(dim=-1, keepdim=True).values  # 取每个位置最大 logit 消除指数溢出。
    shifted_logits = logits - row_maximum  # 平移 logits 且不改变归一化概率。
    log_normalizer = torch.log(torch.exp(shifted_logits).sum(dim=-1, keepdim=True))  # 计算平移后的对数归一化常数。
    return shifted_logits - log_normalizer  # 返回每个词表项的归一化对数概率。
def sequence_log_probability(logits, token_ids, mask, length_normalize=True):  # 根据 token 与 mask 汇总逐条响应对数概率。
    token_log_probabilities = manual_log_softmax(logits).gather(-1, token_ids.unsqueeze(-1)).squeeze(-1)  # 取出实际响应 token 的对数概率。
    masked_log_probabilities = token_log_probabilities * mask.to(logits.dtype)  # 清零 padding 位置的概率贡献。
    summed_log_probabilities = masked_log_probabilities.sum(dim=-1)  # 对每条响应的有效 token 对数概率求和。
    valid_lengths = mask.sum(dim=-1).clamp_min(1).to(logits.dtype)  # 计算有效长度并防止空响应除零。
    return summed_log_probabilities / valid_lengths if length_normalize else summed_log_probabilities  # 按配置返回长度均值或总和。
uniform_logits = torch.zeros(4, 3, 7, dtype=default_dtype, device=device)  # 构造每个词表项等概率的理论样本。
uniform_average = sequence_log_probability(uniform_logits, chosen_tokens, response_mask, length_normalize=True)  # 计算长度归一化序列对数概率。
uniform_sum = sequence_log_probability(uniform_logits, chosen_tokens, response_mask, length_normalize=False)  # 计算未归一化的序列对数概率总和。
expected_uniform = -math.log(7.0)  # 计算七分类均匀分布中单 token 的理论对数概率。
assert torch.allclose(uniform_average, torch.full((4,), expected_uniform, dtype=default_dtype))  # 验证长度均值不会偏向长响应。
assert torch.allclose(uniform_sum[0], torch.tensor(2.0 * expected_uniform, dtype=default_dtype))  # 验证两 token 响应的求和结果。
assert torch.allclose(uniform_sum[1], torch.tensor(3.0 * expected_uniform, dtype=default_dtype))  # 验证三 token 响应的求和结果。
assert torch.all(uniform_average <= 0.0)  # 验证合法序列对数概率不会大于零。
assert torch.isfinite(uniform_average).all()  # 验证手写归一化与 mask 没有产生非数值。

## 5. 手写组合损失并拆开记录

组合函数先对同一模型输出分别收集 chosen 与 rejected token 的 log probability，再按有效长度归一化成逐样本序列 log probability。`chosen_nll` 只使用 chosen；`preference_loss` 使用双方的 log-odds margin；`total_loss` 以系数 `λ` 合并两项。`λ=0` 时退化成普通 chosen SFT，这给出了必要的消融基线。

不要只把 total loss 打到监控面板。至少还应记录 chosen NLL、OR penalty、平均 margin、pair accuracy、有效 token 数和各长度桶统计。若 λ 过大，排序可能迅速变强但生成质量或校准恶化；若 λ 太小，训练行为几乎等同 SFT。

In [ ]:
def orpo_objective(model, beta):  # 计算同一策略模型上的 chosen SFT 与 odds-ratio 组合目标。
    logits = model(prompt_ids, sequence_length=chosen_tokens.shape[1])  # 对当前偏好批次执行一次策略模型前向传播。
    chosen_log_probability = sequence_log_probability(logits, chosen_tokens, response_mask)  # 汇总 chosen 响应的长度归一化对数概率。
    rejected_log_probability = sequence_log_probability(logits, rejected_tokens, response_mask)  # 汇总 rejected 响应的长度归一化对数概率。
    chosen_nll = -chosen_log_probability.mean()  # 计算维持生成学习信号的 chosen 监督负对数似然。
    per_pair_penalty = odds_ratio_penalty(chosen_log_probability, rejected_log_probability)  # 计算每个偏好对的 odds-ratio 惩罚。
    preference_loss = per_pair_penalty.mean()  # 对批次内偏好惩罚求均值。
    total_loss = chosen_nll + beta * preference_loss  # 在单阶段中加权合并监督与偏好两个目标。
    margin = log_odds_from_log_probability(chosen_log_probability) - log_odds_from_log_probability(rejected_log_probability)  # 计算用于诊断的逐对 margin。
    return {"total": total_loss, "chosen_nll": chosen_nll, "preference": preference_loss, "margin": margin, "chosen_logp": chosen_log_probability, "rejected_logp": rejected_log_probability}  # 返回可审计的损失分解。
zero_beta_metrics = orpo_objective(template_model, beta=0.0)  # 计算退化为普通 SFT 的基线目标。
orpo_probe_metrics = orpo_objective(template_model, beta=0.3)  # 计算带偏好约束的 ORPO 探针目标。
assert torch.allclose(zero_beta_metrics["total"], zero_beta_metrics["chosen_nll"])  # 验证 λ 为零时组合目标严格退化为 chosen SFT。
assert orpo_probe_metrics["total"] > orpo_probe_metrics["chosen_nll"]  # 验证正权重会加入非负偏好惩罚。
assert orpo_probe_metrics["preference"] > 0.0  # 验证 logistic 偏好损失始终为正。
assert orpo_probe_metrics["margin"].shape == torch.Size([4])  # 验证每条偏好样本都有独立 margin。
assert torch.all(orpo_probe_metrics["chosen_logp"] <= 0.0)  # 验证 chosen 序列对数概率满足合法范围。
assert torch.all(orpo_probe_metrics["rejected_logp"] <= 0.0)  # 验证 rejected 序列对数概率满足合法范围。

## 6. 面试里一定要讲清梯度方向

若 chosen 与 rejected 的 logit 分别可训练，正确的偏好梯度应满足：chosen logit 的梯度为负，梯度下降会把它抬高；rejected logit 的梯度为正，梯度下降会把它压低。由于两类 softmax 对整体平移不敏感，两者梯度和应接近零。这个单样本梯度探针比只看最终准确率更早发现 chosen/rejected 颠倒、margin 符号写反或 mask 错位。

在完整模型中，一个 token 也会通过共享参数影响其他 prompt 和 token，所以不能把局部梯度方向等价为全局行为。训练中还应监控梯度范数、不同层更新比例、异常样本和 held-out 分布上的漂移。

In [ ]:
pair_logits = torch.tensor([[0.2, -0.1]], dtype=default_dtype, requires_grad=True)  # 创建只含 chosen 与 rejected 两类的可微 logits。
pair_log_probabilities = manual_log_softmax(pair_logits)  # 把二分类 logits 转为稳定对数概率。
scalar_chosen_logp = pair_log_probabilities[:, 0]  # 选取第一类作为 chosen 对数概率。
scalar_rejected_logp = pair_log_probabilities[:, 1]  # 选取第二类作为 rejected 对数概率。
scalar_sft = -scalar_chosen_logp.mean()  # 计算该样本的 chosen 监督负对数似然。
scalar_preference = odds_ratio_penalty(scalar_chosen_logp, scalar_rejected_logp).mean()  # 计算同一样本的 odds-ratio 偏好项。
scalar_total = scalar_sft + 0.4 * scalar_preference  # 合并成一次反向传播使用的 ORPO 目标。
scalar_total.backward()  # 通过自动微分取得两个 logits 的真实梯度方向。
chosen_gradient = float(pair_logits.grad[0, 0].item())  # 读取 chosen logit 的梯度。
rejected_gradient = float(pair_logits.grad[0, 1].item())  # 读取 rejected logit 的梯度。
assert chosen_gradient < 0.0  # 验证梯度下降会提高 chosen logit。
assert rejected_gradient > 0.0  # 验证梯度下降会降低 rejected logit。
assert abs(chosen_gradient + rejected_gradient) < 1e-12  # 验证 softmax 对共同平移保持不变。
assert torch.isfinite(pair_logits.grad).all()  # 验证组合损失的反向传播没有非数值。
assert scalar_total > scalar_sft  # 验证正权重偏好项确实进入总目标。
assert scalar_preference < math.log(2.0)  # 验证当前 chosen 已占优时偏好惩罚低于随机边界。

## 7. 单阶段训练循环与公平消融

下面不用 Trainer，也不用封装好的偏好损失：每步显式清梯度、计算组合目标、反向传播、裁剪梯度并用基础张量更新参数。SFT 与 ORPO 都从 `initial_state` 开始，使用同样步数与学习率，唯一差别是 λ。只有这种控制变量比较，才能把额外 margin 归因于偏好项。

真实训练通常还需梯度累积、混合精度、分布式同步、断点恢复和学习率调度。恢复时不只加载模型权重，还要保存优化器、调度器、数据游标、随机数状态和 λ；否则“继续训练”会改变实验含义。

In [ ]:
def train_candidate(beta, steps=80, learning_rate=0.08):  # 定义不依赖 Trainer 的可复现候选模型训练循环。
    model = TinyLM(prompt_count=4, vocabulary_size=7, hidden_size=12, maximum_length=3).to(device)  # 创建结构一致的新策略模型。
    model.load_state_dict(copy.deepcopy(initial_state))  # 恢复同一初始化以隔离目标函数差异。
    history = []  # 初始化每步训练账本用于检查收敛过程。
    for step in range(steps):  # 按固定更新步数执行单阶段优化。
        model.zero_grad(set_to_none=True)  # 清除上一步梯度并避免无效零值写入。
        metrics = orpo_objective(model, beta=beta)  # 同时计算 chosen SFT 与加权偏好目标。
        metrics["total"].backward()  # 对组合目标执行一次反向传播。
        squared_norm = sum((parameter.grad.detach() ** 2).sum() for parameter in model.parameters())  # 汇总所有参数的梯度平方和。
        gradient_norm = torch.sqrt(squared_norm)  # 计算更新前全局梯度二范数。
        clip_scale = torch.clamp(1.0 / (gradient_norm + 1e-12), max=1.0)  # 计算最大范数为一的手工裁剪比例。
        with torch.no_grad():  # 关闭参数更新本身的梯度记录。
            for parameter in model.parameters():  # 遍历所有模型参数执行显式随机梯度下降。
                parameter.add_(parameter.grad, alpha=-learning_rate * float(clip_scale.item()))  # 使用裁剪后梯度更新当前参数。
        history.append({"step": step, "total": float(metrics["total"].item()), "nll": float(metrics["chosen_nll"].item()), "preference": float(metrics["preference"].item()), "grad_norm": float(gradient_norm.item())})  # 保存分解损失与稳定性信号。
    return model, history  # 返回训练完成的策略模型与完整账本。
sft_model, sft_history = train_candidate(beta=0.0)  # 训练只使用 chosen NLL 的公平对照模型。
orpo_model, orpo_history = train_candidate(beta=0.4)  # 训练在同一阶段加入 odds-ratio 项的 ORPO 模型。
assert len(sft_history) == len(orpo_history) == 80  # 验证两种策略使用完全相同的更新预算。
assert sft_history[-1]["nll"] < sft_history[0]["nll"]  # 验证纯 SFT 能降低 chosen 监督损失。
assert orpo_history[-1]["nll"] < orpo_history[0]["nll"]  # 验证 ORPO 在拉开偏好时仍学习 chosen 响应。
assert orpo_history[-1]["preference"] < orpo_history[0]["preference"]  # 验证 ORPO 显著降低偏好惩罚。
assert all(math.isfinite(record["total"]) for record in orpo_history)  # 验证整个训练轨迹没有 NaN 或无穷大。
assert all(record["grad_norm"] > 0.0 for record in orpo_history)  # 验证每一步都有真实训练信号。
assert sft_model is not orpo_model  # 验证两种策略没有错误共享同一个可变模型对象。

## 8. 评测不能只看训练 loss

最小离线矩阵至少比较初始化、纯 SFT、ORPO 三列，行包括 chosen NLL、平均 log-odds margin、pair accuracy 和非数值率。这里期望两种训练都降低 chosen NLL，而 ORPO 相比同预算 SFT 获得更大的偏好 margin。发布门禁还应加入 held-out 指令遵循、事实性、安全、长度分桶、语言分桶、拒答率和人工盲评；训练集 pair accuracy 为 100% 只说明模型记住了这个玩具批次。

无需 reference model 的收益主要是少一次参考模型前向和显存占用，也减少参考版本管理；代价是缺少显式锚点，并不意味着模型不会漂移。生产中可以额外监控与基座模型的 KL、通用能力回归和输出分布变化，但这些是评测护栏，不需要重新塞入 ORPO 训练公式。

In [ ]:
@torch.no_grad()  # 关闭评测阶段的梯度记录以避免污染训练状态。
def evaluate_candidate(model):  # 计算候选模型的 chosen 拟合与偏好排序指标。
    metrics = orpo_objective(model, beta=0.4)  # 复用同一概率定义取得可比较的损失分解。
    margin = metrics["margin"]  # 读取每条偏好对的稳定 log-odds margin。
    pair_accuracy = (margin > 0.0).to(torch.float64).mean()  # 计算 chosen 排在 rejected 之前的样本比例。
    return {"chosen_nll": float(metrics["chosen_nll"].item()), "margin": float(margin.mean().item()), "pair_accuracy": float(pair_accuracy.item()), "finite": bool(torch.isfinite(margin).all().item())}  # 返回发布门禁需要的普通数值。
initial_report = evaluate_candidate(template_model)  # 评估训练前随机初始化作为基线。
sft_report = evaluate_candidate(sft_model)  # 评估相同步数的 chosen-only SFT 对照。
orpo_report = evaluate_candidate(orpo_model)  # 评估单阶段 ORPO 候选模型。
release_manifest = {"objective": "chosen_nll_plus_odds_ratio", "beta": 0.4, "reference_model_required": False, "length_normalized": True, "steps": 80, "seed": 243}  # 记录复现实验与审核所需关键配置。
release_allowed = orpo_report["finite"] and orpo_report["pair_accuracy"] == 1.0 and orpo_report["margin"] > sft_report["margin"] and orpo_report["chosen_nll"] < initial_report["chosen_nll"]  # 组合本玩具实验的最小发布门禁。
assert sft_report["chosen_nll"] < initial_report["chosen_nll"]  # 验证普通 SFT 确实改善 chosen 拟合。
assert orpo_report["chosen_nll"] < initial_report["chosen_nll"]  # 验证 ORPO 没有牺牲基本 chosen 学习目标。
assert orpo_report["pair_accuracy"] == 1.0  # 验证 ORPO 在训练配对上全部排序正确。
assert orpo_report["margin"] > initial_report["margin"]  # 验证 ORPO 相比初始化扩大平均偏好边界。
assert orpo_report["margin"] > sft_report["margin"]  # 验证加入 odds-ratio 项比同预算 SFT 获得更大 margin。
assert release_manifest["reference_model_required"] is False  # 验证训练清单明确声明无需参考模型。
assert release_manifest["objective"] == "chosen_nll_plus_odds_ratio"  # 验证清单保存了实际使用的组合目标。
assert release_allowed  # 验证候选模型通过本 Notebook 定义的全部最小门禁。

## 9. 面试回答模板与工程边界

可以按六步回答。第一，数据是一条 prompt 配一个 chosen 和 rejected，先做去重、错位检查、长度与安全分桶。第二，同一策略模型计算双方响应 token 的 log probability，padding 与 prompt token 不计入响应 loss，并明确使用求和还是长度均值。第三，用稳定 `log1mexp` 得到双方 log-odds，再以 `softplus(-(log_odds_w-log_odds_l))` 构造排序惩罚。第四，把它与 chosen NLL 按 `L_SFT + λL_OR` 合并，只对一个策略模型做单阶段反向，因此没有 reference model 前向。第五，用 λ=0 的同初始化 SFT 做消融，检查 chosen NLL、margin、pair accuracy、梯度范数和各数据桶。第六，离线通用能力、安全、事实性和人工偏好都过门禁后才发布。

常见追问也要主动说明：ORPO 的“无需 reference”不等于无需基座 checkpoint 或无需漂移监控；单阶段不等于单一损失；odds 项不能修复错误偏好标签；训练 pair accuracy 不能代表泛化；长短响应若概率定义不一致会引入系统偏差；接近概率一时必须稳定计算 `log(1-p)`。在真实 LLM 中还要解决分布式序列 log-prob 对齐、混合精度、梯度累积、checkpoint 恢复和独立安全评测。本例的 TinyLM 只验证公式、接口、梯度方向与公平消融。